In [1]:
import nest_asyncio
from rich.progress import track

from sdg.configs.generators import TranslatorGeneratorConfig

nest_asyncio.apply()

from rich.console import Console

console = Console()

In [2]:
import polars as pl

to_process_en = {
    "drama": [
        # "drie015",
        "lois006",
        "peen001"
    ],
    "jeugdliteratuur": [
        "sche039"
    ],
    "poezie": [
        "lenn006"
    ],
    "proza": [
        "_kle007",
        "_kon002",
        "pier003"
    ]
}

to_process_nl = {
    "jeugdliteratuur": [
        "goej001",
        "hoff049"
    ],
    "poezie": [
        "beer008",
        "dool003"
    ],
}

to_process = to_process_en

source_language = "english"
target_language = "Nederlands"

DATASET_PATH = f"data/qa/selected_books_v2"
SAVE_PATH = DATASET_PATH

In [3]:
def process(author, genre, language):
    import os

    dataset_genre_path = os.path.join(DATASET_PATH, genre)
    dataset_language_path = os.path.join(dataset_genre_path, source_language)
    dataset_author_path = os.path.join(dataset_language_path, f"qa_{author}.parquet")
    df = pl.read_parquet(dataset_author_path)

    from sdg.configs import ModelConfig
    from sdg.generator import TranslationGenerator

    translator_config = TranslatorGeneratorConfig(
        language=language
    )

    model_config = ModelConfig(
        model_name="gpt-4o-mini",
        temperature=0.1,
        timeout=300,
        max_retries=3
    )

    translator = TranslationGenerator(translator_config, model_config)

    ids = []
    questions = []
    answers = []
    genre_list = []
    author_list = []

    for row in track(df.iter_rows(named=True), total=len(df), description="Translating Chunks...", transient=True):
        translated_question = None
        translated_answer = None

        try:
            ids.append(row["ti_id"])
            og_question = row["question"]
            og_answer = row["answer"]
            translated_question = translator.generate_with_splitting(og_question, 10000)
            translated_answer = translator.generate_with_splitting(og_answer, 10000)
        except Exception:
            console.print_exception(show_locals=True)

        if translated_question is None or translated_answer is None or not translated_question.translated_text or not translated_answer.translated_text:
            console.print(f"{row['ti_id']} failed to translate.")
            continue

        questions.append(translated_question.translated_text)
        answers.append(translated_answer.translated_text)
        genre_list.append(genre)
        author_list.append(author)

    new_df = pl.DataFrame({
        "ti_id": ids,
        "author": author_list,
        "genre": genre_list,
        "question": questions,
        "answer": answers
    })

    save_genre_path = os.path.join(SAVE_PATH, genre)
    save_language_path = os.path.join(save_genre_path, language)
    save_author_path = os.path.join(save_language_path, f"qa_{author}.parquet")

    os.makedirs(save_language_path, exist_ok=True)
    new_df.write_parquet(save_author_path)

In [4]:
failed_count = 0
failed_info = {}

for genre, authors in track(to_process.items(), total=len(to_process.items()), description=f"Translating QAs for Corpus...", transient=True):
    for author in track(authors, total=len(authors), description=f"{genre}...", transient=True):
        try:
            process(author, genre, language=target_language)
        except Exception as e:
            console.print_exception()
            failed_count += 1
            if genre not in failed_info:
                failed_info[genre] = {}
            if author not in failed_info[genre]:
                failed_info[genre][author] = {"count": 0, "errors": []}
            failed_info[genre][author]["count"] += 1
            failed_info[genre][author]["errors"].append(str(e))

console.rule("Completed")

Output()

──────────────────────────────────────────────────── Completed ────────────────────────────────────────────────────

In [5]:
failed_info

{}